In [41]:
# 1. Import all necessary libraries and packages ||
import numpy as np
import pandas as pd
from pathlib import Path

In [42]:
# 2. Read the data files into dataframes from the extracted raw data folder
df_matches = pd.read_csv("../data/raw/matches.csv")
df_teams = pd.read_csv("../data/raw/teams.csv")
df_tournaments = pd.read_csv("../data/raw/tournaments.csv")

print(f"Matches Data is loaded with entries: {df_matches.shape}")
print(f"Teams Data is loaded with entries: {df_teams.shape}")
print(f"Tournaments Data is loaded with entries: {df_tournaments.shape}")

Matches Data is loaded with entries: (1068, 28)
Teams Data is loaded with entries: (537, 16)
Tournaments Data is loaded with entries: (23, 4)


In [43]:
print(f"Matches columns: {df_matches.columns}")
print(f"Teams columns: {df_teams.columns}")
print(f"Tournaments columns: {df_tournaments.columns}")

Matches columns: Index(['id', 'date', 'kickoff', 'stage', 'homeTeam', 'awayTeam', 'homeScore',
       'awayScore', 'result', 'extraTime', 'penalties', 'stadium', 'stadiumId',
       'city', 'attendance', 'referee', 'weather', 'stageNormalized',
       'tournament_year', 'captains', 'goals', 'penaltyShootout', 'matchNo',
       'kickoffUtc', 'timezone', 'homeRef', 'awayRef', 'country'],
      dtype='str')
Teams columns: Index(['name', 'code', 'iso', 'confederation', 'groupStage', 'knockoutPath',
       'finalPosition', 'coach', 'captain', 'squad', 'flag', 'tournament_year',
       '_squadFinalizedBy', 'squadKind', 'squadFetched',
       '_squadIsPreliminary'],
      dtype='str')
Tournaments columns: Index(['year', 'host', 'champion', 'file'], dtype='str')


In [44]:
# # remove the 2026 year records (they are our prediction)
# df_matches = df_matches[
#     df_matches["date"] < 2026
# ].copy()

# df_teams = df_teams[
#     df_teams["tournament_year"] < 2026
# ].copy()

# df_tournaments = df_tournaments[
#     df_tournaments["year"] < 2026
# ].copy()

In [45]:
# print(df_matches["year"].unique())
# print(df_teams["tournament_year"].unique())
# print(df_tournaments["year"].unique())

In [46]:
# 2. Feature Engineering: Combining tables with relevant fields that will enable strong predicting basics

df_matches = df_matches.merge(
    df_tournaments,
    left_on="tournament_year",
    right_on="year",
    how="left"
)
df_matches.head()

,id,date,kickoff,stage,homeTeam,awayTeam,homeScore,awayScore,result,extraTime,...,matchNo,kickoffUtc,timezone,homeRef,awayRef,country,year,host,champion,file
0,1930-001,1930-07-13,15:00,group_1,France,Mexico,4.0,1.0,4-1,False,...,NaN,NaN,NaN,NaN,NaN,NaN,1930,['Uruguay'],Uruguay,tournaments/1930.json
1,1930-002,1930-07-15,16:00,group_1,Argentina,France,1.0,0.0,1-0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,1930,['Uruguay'],Uruguay,tournaments/1930.json
2,1930-003,1930-07-16,14:45,group_1,Chile,Mexico,3.0,0.0,3-0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,1930,['Uruguay'],Uruguay,tournaments/1930.json
3,1930-004,1930-07-19,12:50,group_1,Chile,France,1.0,0.0,1-0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,1930,['Uruguay'],Uruguay,tournaments/1930.json
4,1930-005,1930-07-19,15:00,group_1,Argentina,Mexico,6.0,3.0,6-3,False,...,NaN,NaN,NaN,NaN,NaN,NaN,1930,['Uruguay'],Uruguay,tournaments/1930.json


In [47]:
# 3. Select important features from Teams

team_features = df_teams[
    [
        "name",
        "confederation",
        "finalPosition",
        "coach",
        "captain",
        "tournament_year"
    ]

]
team_features.tail()

,name,confederation,finalPosition,coach,captain,tournament_year
532,Colombia,CONMEBOL,NaN,"{'name': 'Néstor Lorenzo', 'country': 'Argenti...",NaN,2026
533,England,UEFA,NaN,"{'name': 'Thomas Tuchel', 'country': 'Germany'...",NaN,2026
534,Croatia,UEFA,NaN,"{'name': 'Zlatko Dalić', 'country': 'Croatia',...",NaN,2026
535,Ghana,CAF,NaN,"{'name': 'Otto Addo', 'country': 'Ghana', 'bor...",NaN,2026
536,Panama,Concacaf,NaN,"{'name': 'Thomas Christiansen', 'country': 'Sp...",NaN,2026


In [48]:
df_matches.columns


Index(['id', 'date', 'kickoff', 'stage', 'homeTeam', 'awayTeam', 'homeScore',
       'awayScore', 'result', 'extraTime', 'penalties', 'stadium', 'stadiumId',
       'city', 'attendance', 'referee', 'weather', 'stageNormalized',
       'tournament_year', 'captains', 'goals', 'penaltyShootout', 'matchNo',
       'kickoffUtc', 'timezone', 'homeRef', 'awayRef', 'country', 'year',
       'host', 'champion', 'file'],
      dtype='str')

In [49]:
df_tournaments.columns

Index(['year', 'host', 'champion', 'file'], dtype='str')

In [50]:
df_teams.columns

Index(['name', 'code', 'iso', 'confederation', 'groupStage', 'knockoutPath',
       'finalPosition', 'coach', 'captain', 'squad', 'flag', 'tournament_year',
       '_squadFinalizedBy', 'squadKind', 'squadFetched',
       '_squadIsPreliminary'],
      dtype='str')

In [51]:
# Clean the tables with the correct date types and standard timezone
df_matches["date"] = pd.to_datetime(df_matches["date"])

df_matches["kickoffUtc"] = pd.to_datetime(
    df_matches["kickoffUtc"]
)

In [52]:
# Standardize Team Names
df_matches["homeTeam"] = (
    df_matches["homeTeam"]
    .str.strip()
)

df_matches["awayTeam"] = (
    df_matches["awayTeam"]
    .str.strip()
)

df_teams["name"] = (
    df_teams["name"]
    .str.strip()
)


In [53]:
# Create a MAtch result target
def build_target(row):

    if row.homeScore > row.awayScore:
        return 1

    elif row.homeScore == row.awayScore:
        return 0

    return -1


df_matches["target"] = (
    df_matches
    .apply(build_target, axis=1)
)



In [54]:
# create a team match dataset
home_df = pd.DataFrame({

    "team":df_matches["homeTeam"],
    "opponent":df_matches["awayTeam"],
    "goals_for":df_matches["homeScore"],
    "goals_against":df_matches["awayScore"],
    "is_home":1,
    "date":df_matches["date"]

})

away_df = pd.DataFrame({

    "team":df_matches["awayTeam"],
    "opponent":df_matches["homeTeam"],
    "goals_for":df_matches["awayScore"],
    "goals_against":df_matches["homeScore"],
    "is_home":0,
    "date":df_matches["date"]

})

team_matches = pd.concat(
    [home_df, away_df],
    ignore_index=True
)

In [55]:
# Matches sorted in a chronological manner
df_matches = df_matches.sort_values(
    "kickoffUtc"
).reset_index(drop=True)

In [56]:
# # Calculate Elo ratings using historical matches.
# # The Elo rating for each team is computed BEFORE each match,
# # based only on the outcomes of all previously played matches.

# for idx, row in df_matches.iterrows():

#     home = row["homeTeam"]
#     away = row["awayTeam"]

#     # Retrieve each team's historical Elo rating
#     home_elo = elo_ratings.get(home, INITIAL_ELO)
#     away_elo = elo_ratings.get(away, INITIAL_ELO)

#     # Store pre-match Elo ratings
#     home_team_elo.append(home_elo)
#     away_team_elo.append(away_elo)

#     # Calculate the Elo difference
#     home_elo_diff.append(home_elo - away_elo)
#     away_elo_diff.append(away_elo - home_elo)

#     # Estimate the expected outcome from historical Elo ratings
#     expected_home = 1 / (
#         1 + 10 ** ((away_elo - home_elo) / 400)
#     )

#     expected_away = 1 - expected_home

#     # Determine the actual match outcome
#     if row["homeScore"] > row["awayScore"]:
#         actual_home = 1
#         actual_away = 0

#     elif row["homeScore"] < row["awayScore"]:
#         actual_home = 0
#         actual_away = 1

#     else:
#         actual_home = 0.5
#         actual_away = 0.5

#     # Update Elo ratings for future matches
#     elo_ratings[home] = (
#         home_elo +
#         K * (actual_home - expected_home)
#     )

#     elo_ratings[away] = (
#         away_elo +
#         K * (actual_away - expected_away)
#     )

In [57]:
# # add these ratings to df_matches
# df_matches["home_team_elo"] = home_team_elo
# df_matches["away_team_elo"] = away_team_elo

# df_matches["home_elo_diff"] = home_elo_diff
# df_matches["away_elo_diff"] = away_elo_diff


In [58]:
df_matches

,id,date,kickoff,stage,homeTeam,awayTeam,homeScore,awayScore,result,extraTime,...,kickoffUtc,timezone,homeRef,awayRef,country,year,host,champion,file,target
0,2026-001,2026-06-11,13:00,group_a,Mexico,South Africa,NaN,NaN,NaN,False,...,2026-06-11 19:00:00+00:00,mexico_city,Mexico,South Africa,Mexico,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
1,2026-002,2026-06-12,20:00,group_a,Korea Republic,Czechia,NaN,NaN,NaN,False,...,2026-06-12 02:00:00+00:00,mexico_city,Korea Republic,Czechia,Mexico,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
2,2026-003,2026-06-12,15:00,group_b,Canada,Bosnia and Herzegovina,NaN,NaN,NaN,False,...,2026-06-12 19:00:00+00:00,toronto,Canada,Bosnia and Herzegovina,Canada,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
3,2026-004,2026-06-13,18:00,group_d,USA,Paraguay,NaN,NaN,NaN,False,...,2026-06-13 01:00:00+00:00,us_pacific,USA,Paraguay,United States,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
4,2026-008,2026-06-13,12:00,group_b,Qatar,Switzerland,NaN,NaN,NaN,False,...,2026-06-13 19:00:00+00:00,us_pacific,Qatar,Switzerland,United States,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1063,2022-060,2022-12-10,22:00,qf,England,France,1.0,2.0,1-2,False,...,NaT,NaN,NaN,NaN,NaN,2022,['Qatar'],Argentina,tournaments/2022.json,-1
1064,2022-061,2022-12-13,22:00,sf,Argentina,Croatia,3.0,0.0,3-0,False,...,NaT,NaN,NaN,NaN,NaN,2022,['Qatar'],Argentina,tournaments/2022.json,1
1065,2022-062,2022-12-14,22:00,sf,France,Morocco,2.0,0.0,2-0,False,...,NaT,NaN,NaN,NaN,NaN,2022,['Qatar'],Argentina,tournaments/2022.json,1
1066,2022-063,2022-12-17,18:00,thirdPlace,Croatia,Morocco,2.0,1.0,2-1,False,...,NaT,NaN,NaN,NaN,NaN,2022,['Qatar'],Argentina,tournaments/2022.json,1


In [59]:
# # build the team related dataset

# home_df["team_elo"] = df_matches["home_team_elo"]
# home_df["opponent_elo"] = df_matches["away_team_elo"]
# home_df["elo_diff"] = df_matches["home_elo_diff"]

# away_df["team_elo"] = df_matches["away_team_elo"]
# away_df["opponent_elo"] = df_matches["home_team_elo"]
# away_df["elo_diff"] = df_matches["away_elo_diff"]

In [60]:
# Combine both home and away...
team_matches = pd.concat(
    [home_df, away_df],
    ignore_index=True
)

In [61]:
# team_matches[
#     [
#         "team",
#         "opponent",
#         "team_elo",
#         "opponent_elo",
#         "elo_diff"
#     ]
# ].head()

In [62]:
# df_matches[
#     [
#         "home_team_elo",
#         "away_team_elo"
#     ]
# ].tail()

In [63]:
# team_matches["elo_diff"].describe()

In [64]:
# team_matches.groupby("team")["team_elo"].max()\
#     .sort_values(ascending=False)\
#     .head(10)

In [65]:
team_matches["team"].nunique()

97

In [66]:
# latest_elo = (
#     team_matches
#     .sort_values("date")
#     .groupby("team")
#     .last()
# )

# latest_elo[
#     "team_elo"
# ].sort_values(
#     ascending=False
# ).head(15)

In [67]:
# Standardize team names

team_name_map = {
    "West Germany": "Germany",
    "USA": "United States",
    "Korea Republic": "South Korea",
    "IR Iran": "Iran"
}

df_matches["homeTeam"] = (
    df_matches["homeTeam"]
    .replace(team_name_map)
)

df_matches["awayTeam"] = (
    df_matches["awayTeam"]
    .replace(team_name_map)
)

df_teams["name"] = (
    df_teams["name"]
    .replace(team_name_map)
)

In [68]:
TEAM_MAPPING = {
    "West Germany": "Germany"
}

def standardize_team_names(df):
    df = df.copy()

    if "homeTeam" in df.columns:
        df["homeTeam"] = df["homeTeam"].replace(TEAM_MAPPING)

    if "awayTeam" in df.columns:
        df["awayTeam"] = df["awayTeam"].replace(TEAM_MAPPING)

    if "name" in df.columns:
        df["name"] = df["name"].replace(TEAM_MAPPING)

    return df

In [69]:
team_mapping = {
    "West Germany": "Germany"
}

df_matches["homeTeam"] = df_matches["homeTeam"].replace(team_mapping)
df_matches["awayTeam"] = df_matches["awayTeam"].replace(team_mapping)
df_teams["name"] = df_teams["name"].replace(team_mapping)

print(
    (df_matches["homeTeam"] == "West Germany").sum()
)

print(
    (df_matches["awayTeam"] == "West Germany").sum()
)

print(
    (df_teams["name"] == "West Germany").sum()
)

0
0
0


In [70]:
print(df_matches[
    df_matches["homeTeam"].str.contains("Germany", na=False)
]["homeTeam"].unique())

<StringArray>
['Germany', 'East Germany']
Length: 2, dtype: str


In [71]:
print(team_matches[
    team_matches["team"].str.contains("Germany", na=False)
]["team"].unique())

<StringArray>
['Germany', 'West Germany', 'East Germany']
Length: 3, dtype: str


In [72]:
df_matches["homeTeam"] = df_matches["homeTeam"].str.strip()
df_matches["awayTeam"] = df_matches["awayTeam"].str.strip()

df_teams["name"] = df_teams["name"].str.strip()

In [73]:
team_mapping = {
    "West Germany": "Germany"
}

df_matches["homeTeam"] = (
    df_matches["homeTeam"]
    .replace(team_mapping)
)

df_matches["awayTeam"] = (
    df_matches["awayTeam"]
    .replace(team_mapping)
)

df_teams["name"] = (
    df_teams["name"]
    .replace(team_mapping)
)

In [74]:
print(df_matches[
    df_matches["homeTeam"].str.contains(
        "Germany",
        na=False
    )
]["homeTeam"].unique())

<StringArray>
['Germany', 'East Germany']
Length: 2, dtype: str


In [75]:
home_df = pd.DataFrame({
    "team": df_matches["homeTeam"],
    "opponent": df_matches["awayTeam"],
    "goals_for": df_matches["homeScore"],
    "goals_against": df_matches["awayScore"],
    "is_home": 1,
    "date": df_matches["date"]
})

away_df = pd.DataFrame({
    "team": df_matches["awayTeam"],
    "opponent": df_matches["homeTeam"],
    "goals_for": df_matches["awayScore"],
    "goals_against": df_matches["homeScore"],
    "is_home": 0,
    "date": df_matches["date"]
})

team_matches = pd.concat(
    [home_df, away_df],
    ignore_index=True
)

In [76]:
team_matches["opponent"].isna().sum()

np.int64(64)

In [77]:
team_matches[
    team_matches["opponent"].isna()
].head(20)

,team,opponent,goals_for,goals_against,is_home,date
72,NaN,NaN,NaN,NaN,1,2026-06-28
73,NaN,NaN,NaN,NaN,1,2026-06-29
74,NaN,NaN,NaN,NaN,1,2026-06-29
75,NaN,NaN,NaN,NaN,1,2026-06-30
76,NaN,NaN,NaN,NaN,1,2026-06-30
77,NaN,NaN,NaN,NaN,1,2026-06-30
78,NaN,NaN,NaN,NaN,1,2026-07-01
79,NaN,NaN,NaN,NaN,1,2026-07-01
80,NaN,NaN,NaN,NaN,1,2026-07-01
81,NaN,NaN,NaN,NaN,1,2026-07-02


In [78]:
df_matches[
    df_matches["awayTeam"].isna() |
    df_matches["homeTeam"].isna()
]

,id,date,kickoff,stage,homeTeam,awayTeam,homeScore,awayScore,result,extraTime,...,kickoffUtc,timezone,homeRef,awayRef,country,year,host,champion,file,target
72,2026-073,2026-06-28,12:00,r32,NaN,NaN,NaN,NaN,NaN,False,...,2026-06-28 19:00:00+00:00,us_pacific,2A,2B,United States,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
73,2026-076,2026-06-29,12:00,r32,NaN,NaN,NaN,NaN,NaN,False,...,2026-06-29 17:00:00+00:00,us_central,1C,2F,United States,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
74,2026-074,2026-06-29,16:30,r32,NaN,NaN,NaN,NaN,NaN,False,...,2026-06-29 20:30:00+00:00,us_east,1E,3ABCDF,United States,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
75,2026-075,2026-06-30,19:00,r32,NaN,NaN,NaN,NaN,NaN,False,...,2026-06-30 01:00:00+00:00,mexico_city,1F,2C,Mexico,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
76,2026-078,2026-06-30,12:00,r32,NaN,NaN,NaN,NaN,NaN,False,...,2026-06-30 17:00:00+00:00,us_central,2E,2I,United States,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
77,2026-077,2026-06-30,17:00,r32,NaN,NaN,NaN,NaN,NaN,False,...,2026-06-30 21:00:00+00:00,us_east,1I,3CDFGH,United States,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
78,2026-079,2026-07-01,19:00,r32,NaN,NaN,NaN,NaN,NaN,False,...,2026-07-01 01:00:00+00:00,mexico_city,1A,3CEFHI,Mexico,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
79,2026-080,2026-07-01,12:00,r32,NaN,NaN,NaN,NaN,NaN,False,...,2026-07-01 16:00:00+00:00,us_east,1L,3EHIJK,United States,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
80,2026-082,2026-07-01,13:00,r32,NaN,NaN,NaN,NaN,NaN,False,...,2026-07-01 20:00:00+00:00,us_pacific,1G,3AEHIJ,United States,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1
81,2026-081,2026-07-02,17:00,r32,NaN,NaN,NaN,NaN,NaN,False,...,2026-07-02 00:00:00+00:00,us_pacific,1D,3BEFIJ,United States,2026,"['United States', 'Canada', 'Mexico']",NaN,tournaments/2026.json,-1


In [79]:
print(df_matches["homeTeam"].isna().sum())
print(df_matches["awayTeam"].isna().sum())

32
32


In [80]:
print((df_matches["homeTeam"].str.strip() == "").sum())
print((df_matches["awayTeam"].str.strip() == "").sum())

0
0


In [81]:
df_matches.columns


Index(['id', 'date', 'kickoff', 'stage', 'homeTeam', 'awayTeam', 'homeScore',
       'awayScore', 'result', 'extraTime', 'penalties', 'stadium', 'stadiumId',
       'city', 'attendance', 'referee', 'weather', 'stageNormalized',
       'tournament_year', 'captains', 'goals', 'penaltyShootout', 'matchNo',
       'kickoffUtc', 'timezone', 'homeRef', 'awayRef', 'country', 'year',
       'host', 'champion', 'file', 'target'],
      dtype='str')

In [82]:
df_matches[
    df_matches["homeTeam"].isna() |
    df_matches["awayTeam"].isna()
][
    [
        "tournament_year",
        "date",
        "homeTeam",
        "awayTeam",
        "homeScore",
        "awayScore",
        "stageNormalized"
    ]
]

,tournament_year,date,homeTeam,awayTeam,homeScore,awayScore,stageNormalized
72,2026,2026-06-28,NaN,NaN,NaN,NaN,round_of_32
73,2026,2026-06-29,NaN,NaN,NaN,NaN,round_of_32
74,2026,2026-06-29,NaN,NaN,NaN,NaN,round_of_32
75,2026,2026-06-30,NaN,NaN,NaN,NaN,round_of_32
76,2026,2026-06-30,NaN,NaN,NaN,NaN,round_of_32
77,2026,2026-06-30,NaN,NaN,NaN,NaN,round_of_32
78,2026,2026-07-01,NaN,NaN,NaN,NaN,round_of_32
79,2026,2026-07-01,NaN,NaN,NaN,NaN,round_of_32
80,2026,2026-07-01,NaN,NaN,NaN,NaN,round_of_32
81,2026,2026-07-02,NaN,NaN,NaN,NaN,round_of_32


In [83]:
historical_matches = (
    df_matches[
        df_matches["tournament_year"] < 2026
    ]
    .copy()
)

In [84]:
historical_matches

,id,date,kickoff,stage,homeTeam,awayTeam,homeScore,awayScore,result,extraTime,...,kickoffUtc,timezone,homeRef,awayRef,country,year,host,champion,file,target
104,1930-001,1930-07-13,15:00,group_1,France,Mexico,4.0,1.0,4-1,False,...,NaT,NaN,NaN,NaN,NaN,1930,['Uruguay'],Uruguay,tournaments/1930.json,1
105,1930-002,1930-07-15,16:00,group_1,Argentina,France,1.0,0.0,1-0,False,...,NaT,NaN,NaN,NaN,NaN,1930,['Uruguay'],Uruguay,tournaments/1930.json,1
106,1930-003,1930-07-16,14:45,group_1,Chile,Mexico,3.0,0.0,3-0,False,...,NaT,NaN,NaN,NaN,NaN,1930,['Uruguay'],Uruguay,tournaments/1930.json,1
107,1930-004,1930-07-19,12:50,group_1,Chile,France,1.0,0.0,1-0,False,...,NaT,NaN,NaN,NaN,NaN,1930,['Uruguay'],Uruguay,tournaments/1930.json,1
108,1930-005,1930-07-19,15:00,group_1,Argentina,Mexico,6.0,3.0,6-3,False,...,NaT,NaN,NaN,NaN,NaN,1930,['Uruguay'],Uruguay,tournaments/1930.json,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1063,2022-060,2022-12-10,22:00,qf,England,France,1.0,2.0,1-2,False,...,NaT,NaN,NaN,NaN,NaN,2022,['Qatar'],Argentina,tournaments/2022.json,-1
1064,2022-061,2022-12-13,22:00,sf,Argentina,Croatia,3.0,0.0,3-0,False,...,NaT,NaN,NaN,NaN,NaN,2022,['Qatar'],Argentina,tournaments/2022.json,1
1065,2022-062,2022-12-14,22:00,sf,France,Morocco,2.0,0.0,2-0,False,...,NaT,NaN,NaN,NaN,NaN,2022,['Qatar'],Argentina,tournaments/2022.json,1
1066,2022-063,2022-12-17,18:00,thirdPlace,Croatia,Morocco,2.0,1.0,2-1,False,...,NaT,NaN,NaN,NaN,NaN,2022,['Qatar'],Argentina,tournaments/2022.json,1


In [85]:
home_df = pd.DataFrame({
    "team": historical_matches["homeTeam"],
    "opponent": historical_matches["awayTeam"],
    "goals_for": historical_matches["homeScore"],
    "goals_against": historical_matches["awayScore"],
    "is_home": 1,
    "date": historical_matches["date"]
})

away_df = pd.DataFrame({
    "team": historical_matches["awayTeam"],
    "opponent": historical_matches["homeTeam"],
    "goals_for": historical_matches["awayScore"],
    "goals_against": historical_matches["homeScore"],
    "is_home": 0,
    "date": historical_matches["date"]
})

team_matches = pd.concat(
    [home_df, away_df],
    ignore_index=True
)

In [86]:
print(team_matches["team"].isna().sum())
print(team_matches["opponent"].isna().sum())

0
0


In [87]:
# Historical matches only
historical_matches = (
    df_matches[
        (df_matches["tournament_year"] < 2026) &
        (df_matches["homeTeam"].notna()) &
        (df_matches["awayTeam"].notna())
    ]
    .copy()
)

# Sort chronologically
historical_matches = (
    historical_matches
    .sort_values("kickoffUtc")
    .reset_index(drop=True)
)

In [88]:
print(historical_matches["tournament_year"].unique())
print(historical_matches["homeTeam"].isna().sum())
print(historical_matches["awayTeam"].isna().sum())

[1930 1934 1938 1950 1954 1958 1962 1966 1970 1974 1978 1982 1986 1990
 1994 1998 2002 2006 2010 2014 2018 2022]
0
0


In [89]:
INITIAL_ELO = 1500
K = 30

elo_ratings = {}

home_team_elo = []
away_team_elo = []

home_elo_diff = []
away_elo_diff = []

In [90]:
# Calculate Elo ratings using historical matches only

for idx, row in historical_matches.iterrows():

    home = row["homeTeam"]
    away = row["awayTeam"]

    # Get pre-match Elo
    home_elo = elo_ratings.get(home, INITIAL_ELO)
    away_elo = elo_ratings.get(away, INITIAL_ELO)

    # Store pre-match ratings
    home_team_elo.append(home_elo)
    away_team_elo.append(away_elo)

    home_elo_diff.append(home_elo - away_elo)
    away_elo_diff.append(away_elo - home_elo)

    # Expected probabilities
    expected_home = 1 / (
        1 + 10 ** ((away_elo - home_elo) / 400)
    )

    expected_away = 1 - expected_home

    # Actual result
    if row["homeScore"] > row["awayScore"]:
        actual_home = 1
        actual_away = 0

    elif row["homeScore"] < row["awayScore"]:
        actual_home = 0
        actual_away = 1

    else:
        actual_home = 0.5
        actual_away = 0.5

    # Update Elo
    elo_ratings[home] = (
        home_elo +
        K * (actual_home - expected_home)
    )

    elo_ratings[away] = (
        away_elo +
        K * (actual_away - expected_away)
    )

In [91]:
historical_matches["home_team_elo"] = home_team_elo
historical_matches["away_team_elo"] = away_team_elo

historical_matches["home_elo_diff"] = home_elo_diff
historical_matches["away_elo_diff"] = away_elo_diff

In [92]:
print(len(historical_matches))
print(len(home_team_elo))
print(len(away_team_elo))
print(len(home_elo_diff))
print(len(away_elo_diff))

964
964
964
964
964


In [93]:
print(len(historical_matches))
print(len(home_team_elo))

964
964


In [94]:
print("historical_matches:", len(historical_matches))
print("home_team_elo:", len(home_team_elo))
print("away_team_elo:", len(away_team_elo))
print("home_elo_diff:", len(home_elo_diff))
print("away_elo_diff:", len(away_elo_diff))

historical_matches: 964
home_team_elo: 964
away_team_elo: 964
home_elo_diff: 964
away_elo_diff: 964


In [95]:
print(historical_matches.columns.tolist())

['id', 'date', 'kickoff', 'stage', 'homeTeam', 'awayTeam', 'homeScore', 'awayScore', 'result', 'extraTime', 'penalties', 'stadium', 'stadiumId', 'city', 'attendance', 'referee', 'weather', 'stageNormalized', 'tournament_year', 'captains', 'goals', 'penaltyShootout', 'matchNo', 'kickoffUtc', 'timezone', 'homeRef', 'awayRef', 'country', 'year', 'host', 'champion', 'file', 'target', 'home_team_elo', 'away_team_elo', 'home_elo_diff', 'away_elo_diff']


In [96]:
historical_matches = historical_matches.copy().reset_index(drop=True)

historical_matches["home_team_elo"] = home_team_elo
historical_matches["away_team_elo"] = away_team_elo

historical_matches["home_elo_diff"] = home_elo_diff
historical_matches["away_elo_diff"] = away_elo_diff

In [97]:
assert len(historical_matches) == len(home_team_elo)
assert len(historical_matches) == len(away_team_elo)
assert len(historical_matches) == len(home_elo_diff)
assert len(historical_matches) == len(away_elo_diff)

In [98]:
# create a whole team centric dataset
home_df = pd.DataFrame({

    "team": historical_matches["homeTeam"],
    "opponent": historical_matches["awayTeam"],
    "goals_for": historical_matches["homeScore"],
    "goals_against": historical_matches["awayScore"],
    "team_elo": historical_matches["home_team_elo"],
    "opponent_elo": historical_matches["away_team_elo"],
    "elo_diff": historical_matches["home_elo_diff"],
    "date": historical_matches["date"],
    "is_home": 1

})

away_df = pd.DataFrame({

    "team": historical_matches["awayTeam"],
    "opponent": historical_matches["homeTeam"],
    "goals_for": historical_matches["awayScore"],
    "goals_against": historical_matches["homeScore"],
    "team_elo": historical_matches["away_team_elo"],
    "opponent_elo": historical_matches["home_team_elo"],
    "elo_diff": historical_matches["away_elo_diff"],
    "date": historical_matches["date"],
    "is_home": 0

})

team_matches = pd.concat(
    [home_df, away_df],
    ignore_index=True
)

team_matches = (
    team_matches
    .sort_values(["team", "date"])
    .reset_index(drop=True)
)


In [99]:
# Create a match outcome
team_matches["win"] = (
    team_matches["goals_for"] >
    team_matches["goals_against"]
).astype(int)

team_matches["draw"] = (
    team_matches["goals_for"] ==
    team_matches["goals_against"]
).astype(int)

team_matches["goal_difference"] = (
    team_matches["goals_for"] -
    team_matches["goals_against"]
)

In [100]:
# Last 5 match win rate
team_matches["team_win_rate_last_5"] = (

    team_matches

    .groupby("team")["win"]

    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .mean()
    )

)

In [101]:
# Goals scored in the last 5
team_matches["team_goals_scored_last_5"] = (

    team_matches

    .groupby("team")["goals_for"]

    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .mean()
    )

)

In [102]:
# Goals conceded in the last 5 matches
team_matches["team_goals_conceded_last_5"] = (

    team_matches

    .groupby("team")["goals_against"]

    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .mean()
    )

)

In [103]:
# find the goal difference in the previous 5 matches
team_matches["goal_difference_last_5"] = (

    team_matches

    .groupby("team")["goal_difference"]

    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .mean()
    )

)

In [104]:
# Check the missing values and fill them

rolling_cols = [

    "team_win_rate_last_5",

    "team_goals_scored_last_5",

    "team_goals_conceded_last_5",

    "goal_difference_last_5"

]

team_matches[rolling_cols] = (
    team_matches[rolling_cols]
    .fillna(0)
)

In [105]:
# Verify the new features
team_matches[
    team_matches["team"] == "Brazil"
][

    [

        "date",

        "team",

        "goals_for",

        "goals_against",

        "team_win_rate_last_5",

        "team_goals_scored_last_5",

        "team_goals_conceded_last_5",

        "goal_difference_last_5"

    ]

].head(15)

,date,team,goals_for,goals_against,team_win_rate_last_5,team_goals_scored_last_5,team_goals_conceded_last_5,goal_difference_last_5
213,1930-07-14,Brazil,1.0,2.0,0.000000,0.0,0.000000,0.000000
214,1930-07-20,Brazil,4.0,0.0,0.000000,1.0,2.000000,-1.000000
215,1934-05-27,Brazil,1.0,3.0,0.500000,2.5,1.000000,1.500000
216,1938-06-05,Brazil,6.0,5.0,0.333333,2.0,1.666667,0.333333
217,1938-06-12,Brazil,1.0,1.0,0.500000,3.0,2.500000,0.500000
218,1938-06-14,Brazil,2.0,1.0,0.400000,2.6,2.200000,0.400000
219,1938-06-16,Brazil,1.0,2.0,0.600000,2.8,2.000000,0.800000
220,1938-06-19,Brazil,4.0,2.0,0.400000,2.2,2.400000,-0.200000
221,1950-06-24,Brazil,4.0,0.0,0.600000,2.8,2.200000,0.600000
222,1950-06-28,Brazil,2.0,2.0,0.600000,2.4,1.200000,1.200000


In [106]:
# Compare head to head matches and results for the teams
historical_matches = (
    historical_matches
    .sort_values("date")
    .reset_index(drop=True)
)

In [107]:
# get the comparisons head to head
from collections import defaultdict

h2h_stats = defaultdict(
    lambda: {
        "games": 0,
        "home_wins": 0,
        "away_wins": 0,
        "draws": 0,
        "goal_diff": 0
    }
)

h2h_games = []
h2h_win_rate = []
h2h_goal_difference = []

In [108]:
# historical head to head results
for _, row in historical_matches.iterrows():

    home = row["homeTeam"]
    away = row["awayTeam"]

    key = tuple(sorted([home, away]))

    stats = h2h_stats[key]

    # Store pre-match history
    h2h_games.append(
        stats["games"]
    )

    if stats["games"] == 0:
        h2h_win_rate.append(0.5)
        h2h_goal_difference.append(0)

    else:

        if home < away:

            wins = stats["home_wins"]

        else:

            wins = stats["away_wins"]

        h2h_win_rate.append(
            wins / stats["games"]
        )

        h2h_goal_difference.append(
            stats["goal_diff"]
        )

    # Update history

    stats["games"] += 1

    if row["homeScore"] > row["awayScore"]:

        if home < away:
            stats["home_wins"] += 1
        else:
            stats["away_wins"] += 1

    elif row["homeScore"] < row["awayScore"]:

        if home < away:
            stats["away_wins"] += 1
        else:
            stats["home_wins"] += 1

    else:

        stats["draws"] += 1

    stats["goal_diff"] += (
        row["homeScore"] -
        row["awayScore"]
    )

In [109]:
# add to our table the new h2h features
historical_matches[
    "head_to_head_games"
] = h2h_games

historical_matches[
    "head_to_head_win_rate"
] = h2h_win_rate

historical_matches[
    "head_to_head_goal_difference"
] = h2h_goal_difference

In [110]:
historical_matches[
    [
        "homeTeam",
        "awayTeam",
        "head_to_head_games",
        "head_to_head_win_rate",
        "head_to_head_goal_difference"
    ]
].tail(15)

,homeTeam,awayTeam,head_to_head_games,head_to_head_win_rate,head_to_head_goal_difference
949,Argentina,Australia,0,0.500000,0.0
950,France,Poland,1,0.000000,1.0
951,England,Senegal,0,0.500000,0.0
952,Japan,Croatia,2,0.000000,-1.0
953,Brazil,South Korea,0,0.500000,0.0
954,Morocco,Spain,1,0.000000,0.0
955,Portugal,Switzerland,0,0.500000,0.0
956,Croatia,Brazil,2,0.000000,3.0
957,Netherlands,Argentina,5,0.400000,7.0
958,Morocco,Portugal,2,0.500000,-1.0


In [111]:
# get the tables into a single row per match to get a target dataset
def build_target(row):

    if row["homeScore"] > row["awayScore"]:
        return 1      # Home Win

    elif row["homeScore"] < row["awayScore"]:
        return -1     # Away Win

    return 0          # Draw


historical_matches["target"] = (
    historical_matches
    .apply(build_target, axis=1)
)

In [112]:
# add weights to the different stages in the competition
stage_weights = {

    "GROUP": 1,
    "ROUND OF 16": 2,
    "QUARTER FINAL": 3,
    "SEMI FINAL": 4,
    "THIRD PLACE": 4,
    "FINAL": 5

}

historical_matches["stage_weight"] = (
    historical_matches["stageNormalized"]
    .map(stage_weights)
    .fillna(1)
)

In [113]:
# Combine the rolling features for home matches
home_features = team_matches[
    [
        "team",
        "date",
        "team_win_rate_last_5",
        "team_goals_scored_last_5",
        "team_goals_conceded_last_5",
        "goal_difference_last_5"
    ]
].copy()

home_features.columns = [

    "homeTeam",
    "date",
    "home_win_rate_last_5",
    "home_goals_last_5",
    "home_conceded_last_5",
    "home_goal_diff_last_5"

]

In [114]:
# get the away features
away_features = team_matches[
    [
        "team",
        "date",
        "team_win_rate_last_5",
        "team_goals_scored_last_5",
        "team_goals_conceded_last_5",
        "goal_difference_last_5"
    ]
].copy()

away_features.columns = [

    "awayTeam",
    "date",
    "away_win_rate_last_5",
    "away_goals_last_5",
    "away_conceded_last_5",
    "away_goal_diff_last_5"

]

In [115]:
# combine the training features together
training_dataset = historical_matches.merge(

    home_features,

    on=["homeTeam", "date"],

    how="left"

)

training_dataset = training_dataset.merge(

    away_features,

    on=["awayTeam", "date"],

    how="left"

)

In [116]:
# separate the features and target for the training
feature_columns = [

    "home_team_elo",
    "away_team_elo",
    "home_elo_diff",

    "home_win_rate_last_5",
    "away_win_rate_last_5",

    "home_goals_last_5",
    "away_goals_last_5",

    "home_conceded_last_5",
    "away_conceded_last_5",

    "home_goal_diff_last_5",
    "away_goal_diff_last_5",

    "head_to_head_games",
    "head_to_head_win_rate",
    "head_to_head_goal_difference",

    "stage_weight"

]

X = training_dataset[feature_columns]

y = training_dataset["target"]

In [117]:
# check dataset
# print(X.shape)

# print(X.head())

print(X.value_counts())

home_team_elo  away_team_elo  home_elo_diff  home_win_rate_last_5  away_win_rate_last_5  home_goals_last_5  away_goals_last_5  home_conceded_last_5  away_conceded_last_5  home_goal_diff_last_5  away_goal_diff_last_5  head_to_head_games  head_to_head_win_rate  head_to_head_goal_difference  stage_weight
1500.000000    1500.000000     0.000000      0.0                   0.0                   0.0                0.0                0.0                   0.0                   0.0                     0.0                   0                   0.500000                0.0                          1.0             8
               1515.000000    -15.000000     0.0                   1.0                   0.0                4.0                0.0                   1.0                   0.0                     3.0                   0                   0.500000                0.0                          1.0             1
               1485.000000     15.000000     0.0                   0.0           

In [118]:
print(
    sorted(
        df_teams["finalPosition"]
        .dropna()
        .unique()
    )
)

[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(28.0), np.float64(29.0), np.float64(30.0), np.float64(31.0), np.float64(32.0)]


In [119]:
# create historical tournament features

df_teams = (
    df_teams
    .sort_values(["name", "tournament_year"])
    .reset_index(drop=True)
)

In [120]:
# build features
team_history = []

for idx, row in df_teams.iterrows():

    team = row["name"]
    year = row["tournament_year"]

    previous = df_teams[
        (df_teams["name"] == team)
        &
        (df_teams["tournament_year"] < year)
    ]

    # Historical titles
    historical_titles = (
        previous["finalPosition"] == 1
    ).sum()

    # Historical knockout rate
    # Top 16 teams reached the knockout stage

    if len(previous) == 0:

        historical_knockout_rate = 0

        historical_average_finish = 32

    else:

        historical_knockout_rate = (
            (
                previous["finalPosition"] <= 16
            ).sum()
            /
            len(previous)
        )

        historical_average_finish = (
            previous["finalPosition"]
            .mean()
        )

    team_history.append({

        "name": team,
        "tournament_year": year,

        "historical_titles":
        historical_titles,

        "historical_knockout_rate":
        historical_knockout_rate,

        "historical_average_finish":
        historical_average_finish

    })

In [121]:
# create the dataframe
historical_team_features = pd.DataFrame(
    team_history
)

In [122]:
training_dataset.columns

Index(['id', 'date', 'kickoff', 'stage', 'homeTeam', 'awayTeam', 'homeScore',
       'awayScore', 'result', 'extraTime', 'penalties', 'stadium', 'stadiumId',
       'city', 'attendance', 'referee', 'weather', 'stageNormalized',
       'tournament_year', 'captains', 'goals', 'penaltyShootout', 'matchNo',
       'kickoffUtc', 'timezone', 'homeRef', 'awayRef', 'country', 'year',
       'host', 'champion', 'file', 'target', 'home_team_elo', 'away_team_elo',
       'home_elo_diff', 'away_elo_diff', 'head_to_head_games',
       'head_to_head_win_rate', 'head_to_head_goal_difference', 'stage_weight',
       'home_win_rate_last_5', 'home_goals_last_5', 'home_conceded_last_5',
       'home_goal_diff_last_5', 'away_win_rate_last_5', 'away_goals_last_5',
       'away_conceded_last_5', 'away_goal_diff_last_5'],
      dtype='str')

In [123]:
# merge home team features
training_dataset = training_dataset.merge(

    historical_team_features,

    left_on=[
        "homeTeam",
        "tournament_year"
    ],

    right_on=[
        "name",
        "tournament_year"
    ],

    how="left"

)

training_dataset.rename(

    columns={

        "historical_titles":
        "home_historical_titles",

        "historical_knockout_rate":
        "home_knockout_rate",

        "historical_average_finish":
        "home_average_finish"

    },

    inplace=True

)

training_dataset.drop(

    columns=[
        "name",
        "tournament_year"
    ],

    inplace=True

)

In [124]:
training_dataset.columns

Index(['id', 'date', 'kickoff', 'stage', 'homeTeam', 'awayTeam', 'homeScore',
       'awayScore', 'result', 'extraTime', 'penalties', 'stadium', 'stadiumId',
       'city', 'attendance', 'referee', 'weather', 'stageNormalized',
       'captains', 'goals', 'penaltyShootout', 'matchNo', 'kickoffUtc',
       'timezone', 'homeRef', 'awayRef', 'country', 'year', 'host', 'champion',
       'file', 'target', 'home_team_elo', 'away_team_elo', 'home_elo_diff',
       'away_elo_diff', 'head_to_head_games', 'head_to_head_win_rate',
       'head_to_head_goal_difference', 'stage_weight', 'home_win_rate_last_5',
       'home_goals_last_5', 'home_conceded_last_5', 'home_goal_diff_last_5',
       'away_win_rate_last_5', 'away_goals_last_5', 'away_conceded_last_5',
       'away_goal_diff_last_5', 'home_historical_titles', 'home_knockout_rate',
       'home_average_finish'],
      dtype='str')

In [126]:
historical_team_features.columns

Index(['name', 'tournament_year', 'historical_titles',
       'historical_knockout_rate', 'historical_average_finish'],
      dtype='str')

In [127]:
# merge away team features
training_dataset = training_dataset.merge(

    historical_team_features,

    left_on=[
        "awayTeam",
        "year"
    ],

    right_on=[
        "name",
        "tournament_year"
    ],

    how="left"

)

training_dataset.rename(

    columns={

        "historical_titles":
        "away_historical_titles",

        "historical_knockout_rate":
        "away_knockout_rate",

        "historical_average_finish":
        "away_average_finish"

    },

    inplace=True

)

training_dataset.drop(

    columns=[
        "name",
        "tournament_year"
    ],

    inplace=True

)

In [128]:
training_dataset["year"] = (
    pd.to_datetime(
        training_dataset["date"]
    ).dt.year
)

In [129]:
print(
    [c for c in training_dataset.columns
     if "historical" in c]
)

print(
    [c for c in training_dataset.columns
     if "knockout" in c]
)

print(
    [c for c in training_dataset.columns
     if "average_finish" in c]
)

['home_historical_titles', 'away_historical_titles']
['home_knockout_rate', 'away_knockout_rate']
['home_average_finish', 'away_average_finish']


In [130]:
# build hte historical best finish
team_history = []

for _, row in df_teams.iterrows():

    team = row["name"]
    year = row["tournament_year"]

    previous = df_teams[
        (df_teams["name"] == team)
        &
        (df_teams["tournament_year"] < year)
    ]

    # Titles

    historical_titles = (
        previous["finalPosition"] == 1
    ).sum()

    # Knockout rate

    if len(previous) == 0:

        historical_knockout_rate = 0
        historical_average_finish = 32

        historical_best_finish = 32
        historical_tournaments_played = 0

    else:

        historical_knockout_rate = (

            (previous["finalPosition"] <= 16).sum()

            /

            len(previous)

        )

        historical_average_finish = (
            previous["finalPosition"]
            .mean()
        )

        # NEW FEATURES

        historical_best_finish = (
            previous["finalPosition"]
            .min()
        )

        historical_tournaments_played = (
            len(previous)
        )

    team_history.append({

        "name": team,

        "tournament_year": year,

        "historical_titles":
        historical_titles,

        "historical_knockout_rate":
        historical_knockout_rate,

        "historical_average_finish":
        historical_average_finish,

        "historical_best_finish":
        historical_best_finish,

        "historical_tournaments_played":
        historical_tournaments_played

    })

In [131]:
#create the feature dataframe
historical_team_features = pd.DataFrame(
    team_history
)

In [132]:
historical_team_features.head(10)

,name,tournament_year,historical_titles,historical_knockout_rate,historical_average_finish,historical_best_finish,historical_tournaments_played
0,Algeria,1982,0,0.000000,32.000000,32.0,0
1,Algeria,1986,0,1.000000,14.000000,14.0,1
2,Algeria,2010,0,0.500000,18.000000,14.0,2
3,Algeria,2014,0,0.333333,21.666667,14.0,3
4,Algeria,2026,0,0.500000,19.250000,12.0,4
5,Angola,2006,0,0.000000,32.000000,32.0,0
6,Argentina,1930,0,0.000000,32.000000,32.0,0
7,Argentina,1934,0,1.000000,2.000000,2.0,1
8,Argentina,1958,0,1.000000,5.500000,2.0,2
9,Argentina,1962,0,1.000000,8.000000,2.0,3


In [133]:
#create a dedicated lookup table
home_history = historical_team_features.copy()

home_history.columns = [

    "homeTeam",

    "year",

    "home_historical_titles",

    "home_knockout_rate",

    "home_average_finish",

    "home_best_finish",

    "home_tournaments_played"

]

In [134]:
# merge to home features
training_dataset = training_dataset.merge(

    home_history,

    on=[
        "homeTeam",
        "year"
    ],

    how="left"

)

In [135]:
#merge for away team
away_history = historical_team_features.copy()

away_history.columns = [

    "awayTeam",

    "year",

    "away_historical_titles",

    "away_knockout_rate",

    "away_average_finish",

    "away_best_finish",

    "away_tournaments_played"

]

training_dataset = training_dataset.merge(

    away_history,

    on=[
        "awayTeam",
        "year"
    ],

    how="left"

)

How I approached the predicting:

-> Training Data:
   1930 - 2018 World Cups

-> Test Data:
   2022 World Cup

In [136]:
train = training_dataset[
    training_dataset["year"] <= 2018
]

test = training_dataset[
    training_dataset["year"] == 2022
]

In [137]:
print(train.shape)
print(test.shape)

(900, 64)
(64, 64)


In [138]:
X_train = train[feature_columns]

y_train = train["target"]

X_test = test[feature_columns]

y_test = test["target"]

In [139]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (900, 15)
y_train: (900,)
X_test : (64, 15)
y_test : (64,)


In [140]:
print(X_train.isnull().sum())

print(X_test.isnull().sum())

home_team_elo                   0
away_team_elo                   0
home_elo_diff                   0
home_win_rate_last_5            0
away_win_rate_last_5            0
home_goals_last_5               0
away_goals_last_5               0
home_conceded_last_5            0
away_conceded_last_5            0
home_goal_diff_last_5           0
away_goal_diff_last_5           0
head_to_head_games              0
head_to_head_win_rate           0
head_to_head_goal_difference    0
stage_weight                    0
dtype: int64
home_team_elo                   0
away_team_elo                   0
home_elo_diff                   0
home_win_rate_last_5            0
away_win_rate_last_5            0
home_goals_last_5               0
away_goals_last_5               0
home_conceded_last_5            0
away_conceded_last_5            0
home_goal_diff_last_5           0
away_goal_diff_last_5           0
head_to_head_games              0
head_to_head_win_rate           0
head_to_head_goal_difference    0
s

In [141]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    verbose=100
)

model.fit(
    X_train,
    y_train
)

0:	learn: 1.0838731	total: 53.5ms	remaining: 26.7s
100:	learn: 0.8080918	total: 467ms	remaining: 1.84s
200:	learn: 0.7040054	total: 869ms	remaining: 1.29s
300:	learn: 0.6213334	total: 1.3s	remaining: 860ms
400:	learn: 0.5443865	total: 1.7s	remaining: 420ms
499:	learn: 0.4821741	total: 2.09s	remaining: 0us


CatBoostClassifier(depth=6, iterations=500, learning_rate=0.05, loss_function='MultiClass', verbose=100)

In [142]:
predictions = model.predict(X_test)

probabilities = model.predict_proba(X_test)

In [143]:
results = test[
    [
        "homeTeam",
        "awayTeam",
        "homeScore",
        "awayScore",
        "target"
    ]
].copy()

results["prediction"] = predictions

In [144]:
model.classes_

[-1, 0, 1]

[-1, 0, 1]

In [145]:
results["away_win_probability"] = probabilities[:,0]

results["draw_probability"] = probabilities[:,1]

results["home_win_probability"] = probabilities[:,2]

In [146]:
prediction_map = {

    1: "Home Win",

    0: "Draw",

    -1: "Away Win"

}

results["predicted_result"] = (
    results["prediction"]
    .map(prediction_map)
)

results["actual_result"] = (
    results["target"]
    .map(prediction_map)
)

In [147]:
results.head(20)

,homeTeam,awayTeam,homeScore,awayScore,target,prediction,away_win_probability,draw_probability,home_win_probability,predicted_result,actual_result
900,Qatar,Ecuador,0.0,2.0,-1,-1,0.449523,0.146828,0.403648,Away Win,Away Win
901,Senegal,Netherlands,0.0,2.0,-1,-1,0.668225,0.220263,0.111512,Away Win,Away Win
902,England,Iran,6.0,2.0,1,1,0.234697,0.200188,0.565115,Home Win,Home Win
903,United States,Wales,1.0,1.0,0,-1,0.421591,0.221800,0.356609,Away Win,Draw
904,France,Australia,4.0,1.0,1,1,0.035239,0.095079,0.869681,Home Win,Home Win
905,Denmark,Tunisia,0.0,0.0,0,1,0.223246,0.209633,0.567122,Home Win,Draw
906,Argentina,Saudi Arabia,1.0,2.0,-1,1,0.059912,0.157739,0.782349,Home Win,Away Win
907,Mexico,Poland,0.0,0.0,0,1,0.172360,0.214232,0.613408,Home Win,Draw
908,Germany,Japan,1.0,2.0,-1,1,0.055039,0.132806,0.812156,Home Win,Away Win
909,Spain,Costa Rica,7.0,0.0,1,1,0.119027,0.124109,0.756865,Home Win,Home Win


In [148]:
wrong_predictions = results[
    results["actual_result"]
    !=
    results["predicted_result"]
]

len(wrong_predictions)

28

In [149]:
results["confidence"] = results[
    [
        "home_win_probability",
        "draw_probability",
        "away_win_probability"
    ]
].max(axis=1)

results.sort_values(
    "confidence",
    ascending=False
).head(20)

,homeTeam,awayTeam,homeScore,awayScore,target,prediction,away_win_probability,draw_probability,home_win_probability,predicted_result,actual_result,confidence
911,Belgium,Canada,1.0,0.0,1,1,0.048920,0.051507,0.899573,Home Win,Home Win,0.899573
923,Argentina,Mexico,2.0,0.0,1,1,0.047194,0.078330,0.874476,Home Win,Home Win,0.874476
904,France,Australia,4.0,1.0,1,1,0.035239,0.095079,0.869681,Home Win,Home Win,0.869681
950,France,Poland,3.0,1.0,1,1,0.042843,0.096063,0.861094,Home Win,Home Win,0.861094
908,Germany,Japan,1.0,2.0,-1,1,0.055039,0.132806,0.812156,Home Win,Away Win,0.812156
919,England,United States,0.0,0.0,0,1,0.070732,0.118970,0.810298,Home Win,Draw,0.810298
915,Portugal,Ghana,3.0,2.0,1,1,0.131034,0.067930,0.801037,Home Win,Home Win,0.801037
927,Croatia,Canada,4.0,1.0,1,1,0.098266,0.112921,0.788813,Home Win,Home Win,0.788813
906,Argentina,Saudi Arabia,1.0,2.0,-1,1,0.059912,0.157739,0.782349,Home Win,Away Win,0.782349
913,Brazil,Serbia,2.0,0.0,1,1,0.085733,0.152844,0.761423,Home Win,Home Win,0.761423


In [150]:
results.to_csv(
    "worldcup_predictions.csv",
    index=False
)

In [151]:
importance = pd.DataFrame({

    "Feature": X_train.columns,

    "Importance":
    model.get_feature_importance()

})

importance.sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
1,away_team_elo,11.670672
2,home_elo_diff,11.170201
0,home_team_elo,10.416198
7,home_conceded_last_5,9.932024
5,home_goals_last_5,9.452458
6,away_goals_last_5,9.141224
8,away_conceded_last_5,8.153055
10,away_goal_diff_last_5,6.259428
9,home_goal_diff_last_5,5.975848
3,home_win_rate_last_5,5.646794


In [152]:
results = pd.DataFrame({

    "Home Team":
    test["homeTeam"],

    "Away Team":
    test["awayTeam"],

    "Actual":
    test["target"].map(prediction_map),

    "Predicted":
    predictions.flatten()

})

results["Predicted"] = (
    results["Predicted"]
    .map(prediction_map)
)

results["Home Win %"] = (
    probabilities[:,2] * 100
)

results["Draw %"] = (
    probabilities[:,1] * 100
)

results["Away Win %"] = (
    probabilities[:,0] * 100
)

results

,Home Team,Away Team,Actual,Predicted,Home Win %,Draw %,Away Win %
900,Qatar,Ecuador,Away Win,Away Win,40.364842,14.682816,44.952342
901,Senegal,Netherlands,Away Win,Away Win,11.151232,22.026256,66.822513
902,England,Iran,Home Win,Home Win,56.511504,20.018785,23.469711
903,United States,Wales,Draw,Away Win,35.660888,22.180043,42.159069
904,France,Australia,Home Win,Home Win,86.968142,9.507914,3.523944
...,...,...,...,...,...,...,...
959,England,France,Away Win,Away Win,25.521418,13.384234,61.094349
960,Argentina,Croatia,Home Win,Home Win,64.750131,20.376842,14.873027
961,France,Morocco,Home Win,Home Win,60.092048,30.007808,9.900145
962,Croatia,Morocco,Home Win,Home Win,70.782571,12.548235,16.669194


In [153]:
def predict_match(
    model,
    match_features
):

    probabilities = model.predict_proba(
        match_features
    )[0]

    classes = model.classes_

    prediction = classes[
        probabilities.argmax()
    ]

    return {

        "prediction": prediction,

        "away_win_probability":
        probabilities[
            list(classes).index(-1)
        ],

        "draw_probability":
        probabilities[
            list(classes).index(0)
        ],

        "home_win_probability":
        probabilities[
            list(classes).index(1)
        ]

    }

In [154]:
def knockout_winner(
    home_team,
    away_team,
    match_features,
    model
):

    result = predict_match(
        model,
        match_features
    )

    if (
        result["home_win_probability"]
        >=
        result["away_win_probability"]
    ):

        return home_team

    return away_team

In [155]:
quarter_pairs = [

    ("", ""),

    ("", ""),

    ("", ""),

    ("", "")

]

In [157]:
def build_match_features(

    home_team,
    away_team,
    stage_weight=1

):

    home_elo = latest_elo.get(
        home_team,
        1500
    )

    away_elo = latest_elo.get(
        away_team,
        1500
    )

    h2h_games, h2h_win_rate, h2h_goal_diff = (

        get_head_to_head(

            home_team,

            away_team

        )

    )

    row = {

        # Elo

        "home_team_elo":
        home_elo,

        "away_team_elo":
        away_elo,

        "home_elo_diff":
        home_elo - away_elo,

        # Rolling Form

        "home_win_rate_last_5":

        rolling_stats
        .get(
            home_team,
            {}
        )
        .get(
            "win_rate",
            0
        ),

        "away_win_rate_last_5":

        rolling_stats
        .get(
            away_team,
            {}
        )
        .get(
            "win_rate",
            0
        ),

        "home_goals_last_5":

        rolling_stats
        .get(
            home_team,
            {}
        )
        .get(
            "goals_scored",
            0
        ),

        "away_goals_last_5":

        rolling_stats
        .get(
            away_team,
            {}
        )
        .get(
            "goals_scored",
            0
        ),

        "home_conceded_last_5":

        rolling_stats
        .get(
            home_team,
            {}
        )
        .get(
            "goals_conceded",
            0
        ),

        "away_conceded_last_5":

        rolling_stats
        .get(
            away_team,
            {}
        )
        .get(
            "goals_conceded",
            0
        ),

        "home_goal_diff_last_5":

        rolling_stats
        .get(
            home_team,
            {}
        )
        .get(
            "goal_difference",
            0
        ),

        "away_goal_diff_last_5":

        rolling_stats
        .get(
            away_team,
            {}
        )
        .get(
            "goal_difference",
            0
        ),

        # Head to Head

        "head_to_head_games":
        h2h_games,

        "head_to_head_win_rate":
        h2h_win_rate,

        "head_to_head_goal_difference":
        h2h_goal_diff,

        # Tournament Experience

        "home_historical_titles":

        latest_history
        .get(
            home_team,
            {}
        )
        .get(
            "titles",
            0
        ),

        "away_historical_titles":

        latest_history
        .get(
            away_team,
            {}
        )
        .get(
            "titles",
            0
        ),

        "home_knockout_rate":

        latest_history
        .get(
            home_team,
            {}
        )
        .get(
            "knockout_rate",
            0
        ),

        "away_knockout_rate":

        latest_history
        .get(
            away_team,
            {}
        )
        .get(
            "knockout_rate",
            0
        ),

        "home_average_finish":

        latest_history
        .get(
            home_team,
            {}
        )
        .get(
            "average_finish",
            32
        ),

        "away_average_finish":

        latest_history
        .get(
            away_team,
            {}
        )
        .get(
            "average_finish",
            32
        ),

        # Tournament Stage

        "stage_weight":
        stage_weight

    }

    return pd.DataFrame(
        [row]
    )

In [170]:
semi_finalists = []

for home, away in quarter_pairs:

    features = build_match_features(
        home,
        away
    )

    winner = knockout_winner(

        home,
        away,
        features,
        model

    )

    semi_finalists.append(
        winner
    )

In [171]:
semi_pairs = [

    (
        semi_finalists[0],
        semi_finalists[1]
    ),

    (
        semi_finalists[2],
        semi_finalists[3]
    )

]

In [172]:
finalists = []

for home, away in semi_pairs:

    features = build_match_features(
        home,
        away
    )

    winner = knockout_winner(

        home,
        away,
        features,
        model

    )

    finalists.append(
        winner
    )

In [173]:
features = build_match_features(

    finalists[0],

    finalists[1]

)

champion = knockout_winner(

    finalists[0],

    finalists[1],

    features,

    model

)

In [175]:
print()

print(
    "Quarter-finalists"
)

print(
    "-----------------"
)

for team in quarter_finalists:

    print(team)

print()

print(
    "Semi-finalists"
)

print(
    "-----------------"
)

for team in semi_finalists:

    print(team)

print()

print(
    "Finalists"
)

print(
    "-----------------"
)

for team in finalists:

    print(team)

print()

print(
    "Champion"
)

print(
    "-----------------"
)

print(
    champion
)


Quarter-finalists
-----------------


NameError: name 'quarter_finalists' is not defined

In [176]:
latest_elo = {}

for team in team_matches["team"].unique():

    latest_elo[team] = (
        team_matches[
            team_matches["team"] == team
        ]
        .sort_values("date")
        ["team_elo"]
        .iloc[-1]
    )

In [177]:
rolling_stats = {}

for team in team_matches["team"].unique():

    recent = (

        team_matches[
            team_matches["team"] == team
        ]

        .sort_values("date")

        .tail(5)

    )

    rolling_stats[team] = {

        "win_rate":

        (
            (
                recent["goals_for"]
                >
                recent["goals_against"]
            ).mean()
        ),

        "goals_scored":

        recent["goals_for"].mean(),

        "goals_conceded":

        recent["goals_against"].mean(),

        "goal_difference":

        (

            recent["goals_for"]

            -

            recent["goals_against"]

        ).mean()

    }

In [178]:
latest_history = {}

for team in historical_team_features["name"].unique():

    latest = (

        historical_team_features[
            historical_team_features["name"]
            == team
        ]

        .sort_values(
            "tournament_year"
        )

        .iloc[-1]

    )

    latest_history[team] = {

        "titles":

        latest[
            "historical_titles"
        ],

        "knockout_rate":

        latest[
            "historical_knockout_rate"
        ],

        "average_finish":

        latest[
            "historical_average_finish"
        ]

    }

In [179]:
def get_head_to_head(
    home_team,
    away_team
):

    games = historical_matches[

        (

            (
                historical_matches[
                    "homeTeam"
                ]
                ==
                home_team
            )

            &

            (
                historical_matches[
                    "awayTeam"
                ]
                ==
                away_team
            )

        )

        |

        (

            (
                historical_matches[
                    "homeTeam"
                ]
                ==
                away_team
            )

            &

            (
                historical_matches[
                    "awayTeam"
                ]
                ==
                home_team
            )

        )

    ]

    if len(games) == 0:

        return 0, 0.5, 0

    home_wins = 0

    goal_diff = 0

    for _, row in games.iterrows():

        if row["homeTeam"] == home_team:

            goal_diff += (

                row["homeScore"]

                -

                row["awayScore"]

            )

            if (

                row["homeScore"]

                >

                row["awayScore"]

            ):

                home_wins += 1

        else:

            goal_diff += (

                row["awayScore"]

                -

                row["homeScore"]

            )

            if (

                row["awayScore"]

                >

                row["homeScore"]

            ):

                home_wins += 1

    return (

        len(games),

        home_wins / len(games),

        goal_diff / len(games)

    )

In [169]:
match = build_match_features(

    "Brazil",

    "Germany",

    stage_weight=5

)

print(match)

   home_team_elo  away_team_elo  home_elo_diff  home_win_rate_last_5  \
0    1701.178671    1684.103438      17.075233                   0.6   

   away_win_rate_last_5  home_goals_last_5  away_goals_last_5  \
0                   0.4                1.6                1.6   

   home_conceded_last_5  away_conceded_last_5  home_goal_diff_last_5  ...  \
0                   0.6                   1.6                    1.0  ...   

   head_to_head_games  head_to_head_win_rate  head_to_head_goal_difference  \
0                   2                    0.5                          -2.0   

   home_historical_titles  away_historical_titles  home_knockout_rate  \
0                       5                       4                 1.0   

   away_knockout_rate  home_average_finish  away_average_finish  stage_weight  
0                 0.9             4.590909                  5.3             5  

[1 rows x 21 columns]


In [180]:
prediction = model.predict(
    match
)

probabilities = model.predict_proba(
    match
)

print(prediction)
print(probabilities)

[[-1]]
[[0.43339737 0.16006571 0.40653693]]


In [181]:
import numpy as np

def simulate_knockout_match(
    home_team,
    away_team,
    stage_weight,
    model
):

    features = build_match_features(

        home_team,
        away_team,
        stage_weight

    )

    probabilities = model.predict_proba(
        features
    )[0]

    classes = model.classes_

    prob_map = dict(
        zip(classes, probabilities)
    )

    home_prob = prob_map.get(1, 0)

    draw_prob = prob_map.get(0, 0)

    away_prob = prob_map.get(-1, 0)

    # Simple knockout logic

    if home_prob >= away_prob:

        winner = home_team

    else:

        winner = away_team

    return {

        "home": home_team,

        "away": away_team,

        "winner": winner,

        "home_win_probability":
        home_prob,

        "draw_probability":
        draw_prob,

        "away_win_probability":
        away_prob

    }

In [182]:
simulate_knockout_match(

    "Brazil",

    "Germany",

    stage_weight=5,

    model=model

)

{'home': 'Brazil',
 'away': 'Germany',
 'winner': 'Germany',
 'home_win_probability': np.float64(0.40653692631925675),
 'draw_probability': np.float64(0.16006570556682592),
 'away_win_probability': np.float64(0.4333973681139174)}

In [183]:
def simulate_round(
    fixtures,
    stage_weight,
    model
):

    winners = []

    for home, away in fixtures:

        result = simulate_knockout_match(

            home,

            away,

            stage_weight,

            model

        )

        winners.append(
            result["winner"]
        )

    return winners

In [184]:
quarter_finals = [

    ("Brazil","Germany"),

    ("France","Portugal"),

    ("Argentina","England"),

    ("Spain","Netherlands")

]

semi_finalists = simulate_round(

    quarter_finals,

    stage_weight=3,

    model=model

)

print(semi_finalists)

['Germany', 'France', 'Argentina', 'Netherlands']


In [185]:
semi_finals = [

    (
        semi_finalists[0],
        semi_finalists[1]
    ),

    (
        semi_finalists[2],
        semi_finalists[3]
    )

]

finalists = simulate_round(

    semi_finals,

    stage_weight=4,

    model=model

)

print(finalists)

['Germany', 'Netherlands']


In [186]:
final_match = [

    (
        finalists[0],
        finalists[1]
    )

]

champion = simulate_round(

    final_match,

    stage_weight=5,

    model=model

)

print(champion)

['Germany']


In [187]:
def simulate_tournament(

    quarter_finals,

    model

):

    semi_finalists = simulate_round(

        quarter_finals,

        stage_weight=3,

        model=model

    )

    semi_finals = [

        (
            semi_finalists[0],
            semi_finalists[1]
        ),

        (
            semi_finalists[2],
            semi_finalists[3]
        )

    ]

    finalists = simulate_round(

        semi_finals,

        stage_weight=4,

        model=model

    )

    final = [

        (
            finalists[0],
            finalists[1]
        )

    ]

    champion = simulate_round(

        final,

        stage_weight=5,

        model=model
    )

    return {

        "Quarter-finalists":

        [team
         for pair in quarter_finals
         for team in pair],

        "Semi-finalists":

        semi_finalists,

        "Finalists":

        finalists,

        "Champion":

        champion[0]

    }

In [188]:
results = simulate_tournament(

    quarter_finals,

    model

)

print()

print(
    "Quarter-finalists"
)

print(
    "-----------------"
)

for team in results[
    "Quarter-finalists"
]:

    print(team)

print()

print(
    "Semi-finalists"
)

print(
    "-----------------"
)

for team in results[
    "Semi-finalists"
]:

    print(team)

print()

print(
    "Finalists"
)

print(
    "-----------------"
)

for team in results[
    "Finalists"
]:

    print(team)

print()

print(
    "Champion"
)

print(
    "-----------------"
)

print(
    results[
        "Champion"
    ]
)


Quarter-finalists
-----------------
Brazil
Germany
France
Portugal
Argentina
England
Spain
Netherlands

Semi-finalists
-----------------
Germany
France
Argentina
Netherlands

Finalists
-----------------
Germany
Netherlands

Champion
-----------------
Germany


In [189]:
y_pred = model.predict(X_test)

In [190]:
y_pred = y_pred.flatten()

In [191]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [192]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print(f"Accuracy: {accuracy:.4f}")

precision = precision_score(

    y_test,

    y_pred,

    average="weighted"

)

print(
    f"Precision: {precision:.4f}"
)

recall = recall_score(

    y_test,

    y_pred,

    average="weighted"

)

print(
    f"Recall: {recall:.4f}"
)

f1 = f1_score(

    y_test,

    y_pred,

    average="weighted"

)

print(
    f"F1 Score: {f1:.4f}"
)

Accuracy: 0.5625
Precision: 0.5448
Recall: 0.5625
F1 Score: 0.5216


In [193]:
print(
    classification_report(
        y_test,
        y_pred,
        labels=[-1,0,1],
        target_names=[
            "Away Win",
            "Draw",
            "Home Win"
        ]
    )
)

              precision    recall  f1-score   support

    Away Win       0.48      0.50      0.49        20
        Draw       0.50      0.13      0.21        15
    Home Win       0.62      0.83      0.71        29

    accuracy                           0.56        64
   macro avg       0.53      0.49      0.47        64
weighted avg       0.54      0.56      0.52        64



In [194]:
importance = pd.DataFrame({

    "Feature":
    X_train.columns,

    "Importance":
    model.get_feature_importance()

})

importance = importance.sort_values(

    "Importance",

    ascending=False

)

print(importance)

                         Feature  Importance
1                  away_team_elo   11.670672
2                  home_elo_diff   11.170201
0                  home_team_elo   10.416198
7           home_conceded_last_5    9.932024
5              home_goals_last_5    9.452458
6              away_goals_last_5    9.141224
8           away_conceded_last_5    8.153055
10         away_goal_diff_last_5    6.259428
9          home_goal_diff_last_5    5.975848
3           home_win_rate_last_5    5.646794
4           away_win_rate_last_5    4.614522
11            head_to_head_games    2.927261
13  head_to_head_goal_difference    2.461752
12         head_to_head_win_rate    2.178562
14                  stage_weight    0.000000
